In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

dbutils.widgets.text("catalogo", "proyecto_ecommerce")
catalogo = dbutils.widgets.get("catalogo")

df_clientes = spark.table(f"{catalogo}.silver.clientes")
df_productos = spark.table(f"{catalogo}.silver.productos")
df_ordenes = spark.table(f"{catalogo}.silver.ordenes")

In [0]:
df_gold_region = spark.table(f"{catalogo}.gold.dim_region")

fact_ventas = (
    df_ordenes
    .join(df_clientes.select("cliente_id", "region"), "cliente_id", "inner")  # excluye los 3 huérfanos
    .join(df_gold_region, "region", "inner")
    .withColumn("fecha_id", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("descuento_monto", F.round(F.col("monto_bruto") - F.col("monto_neto"), 2))
    .select(
        "orden_id", "fecha_id", "cliente_id", "producto_id", "region_id",
        "canal", "cantidad", "precio_unitario", "monto_bruto",
        "descuento_monto", "monto_neto",
    )
)

display(fact_ventas.limit(10))

total_fact = fact_ventas.count()
print(f"gold.fact_ventas (aún sin guardar) -> {total_fact} filas")
print(f"Diferencia vs. silver.ordenes ({df_ordenes.count()}): {df_ordenes.count() - total_fact} excluidas por cliente_id inválido")

In [0]:
(fact_ventas.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.gold.fact_ventas"))

print(f"Guardado: {catalogo}.gold.fact_ventas -> {fact_ventas.count()} filas")